In [1]:
from transformers import TFAutoModel
import tensorflow as tf

In [2]:
bert= TFAutoModel.from_pretrained('bert-base-multilingual-cased', use_safetensors=False)
bert.summary()

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some layers from the model checkpoint at bert-base-multilingual-cased were not used when initializing TFBertModel: ['mlm___cls', 'nsp___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertModel were initialized from the model checkpoint at bert-base-multilingual-cased.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions wi

Model: "tf_bert_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  177853440 
                                                                 
Total params: 177853440 (678.46 MB)
Trainable params: 177853440 (678.46 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [3]:
# two input layers, we ensure layer name variables match to dictionary keys in TF dataset
input_ids = tf.keras.layers.Input(shape=(512,), name='input_ids', dtype='int32')
mask = tf.keras.layers.Input(shape=(512,), name='attention_mask', dtype='int32')

# we access the transformer model within our bert object using the bert attribute (eg bert.bert instead of bert)
embeddings = bert.bert(input_ids, attention_mask=mask)[1]  # access final activations with [0]

# output
x = tf.keras.layers.Dense(1024, activation='relu')(embeddings)
y = tf.keras.layers.Dense(2, activation='softmax', name='outputs')(x)

# initialize model
model = tf.keras.Model(inputs=[input_ids, mask], outputs=y)

# freeze bert layer
model.layers[2].trainable = False

# print out model summary
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_ids (InputLayer)      [(None, 512)]                0         []                            
                                                                                                  
 attention_mask (InputLayer  [(None, 512)]                0         []                            
 )                                                                                                
                                                                                                  
 bert (TFBertMainLayer)      TFBaseModelOutputWithPooli   1778534   ['input_ids[0][0]',           
                             ngAndCrossAttentions(last_   40         'attention_mask[0][0]']      
                             hidden_state=(None, 512, 7                                       

In [6]:
optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)
loss = tf.keras.losses.CategoricalCrossentropy()
acc = tf.keras.metrics.CategoricalAccuracy('accuracy')

model.compile(optimizer=optimizer, loss=loss, metrics=[acc])

element_spec = ({'input_ids': tf.TensorSpec(shape=(16, 512), dtype=tf.int32, name=None),
                 'attention_mask': tf.TensorSpec(shape=(16, 512), dtype=tf.int32, name=None)},
                tf.TensorSpec(shape=(16, 2), dtype=tf.float32, name=None))

# load the training and validation sets
train_ds = tf.data.experimental.load('dataset/second/train', element_spec=element_spec)
val_ds = tf.data.experimental.load('dataset/second/val', element_spec=element_spec)

# view the input format
train_ds.take(1)


Instructions for updating:
Use `tf.data.Dataset.load(...)` instead.


Instructions for updating:
Use `tf.data.Dataset.load(...)` instead.


<_TakeDataset element_spec=({'input_ids': TensorSpec(shape=(16, 512), dtype=tf.int32, name=None), 'attention_mask': TensorSpec(shape=(16, 512), dtype=tf.int32, name=None)}, TensorSpec(shape=(16, 2), dtype=tf.float32, name=None))>

In [7]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=6
)

Epoch 1/6



19/19 [==============================] - 73s 4s/step - loss: 0.7035 - accuracy: 0.4539 - val_loss: 0.6898 - val_accuracy: 0.5312
Epoch 2/6
19/19 [==============================] - 68s 4s/step - loss: 0.6993 - accuracy: 0.5362 - val_loss: 0.6796 - val_accuracy: 0.6250
Epoch 3/6
19/19 [==============================] - 70s 4s/step - loss: 0.6890 - accuracy: 0.5526 - val_loss: 0.6736 - val_accuracy: 0.5938
Epoch 4/6
19/19 [==============================] - 72s 4s/step - loss: 0.6834 - accuracy: 0.5559 - val_loss: 0.6659 - val_accuracy: 0.6562
Epoch 5/6
19/19 [==============================] - 71s 4s/step - loss: 0.6699 - accuracy: 0.6020 - val_loss: 0.6617 - val_accuracy: 0.6250
Epoch 6/6
19/19 [==============================] - 73s 4s/step - loss: 0.6600 - accuracy: 0.6316 - val_loss: 0.6513 - val_accuracy: 0.6562


In [8]:
model.save('lost_dogs_model')

INFO:tensorflow:Assets written to: lost_dogs_model\assets


INFO:tensorflow:Assets written to: lost_dogs_model\assets
